<a href="https://colab.research.google.com/github/cityhunter0831/sar-atr/blob/claude%2Fkind-gauss-84kr3m/notebooks/colab_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAR-ATR Colab 실행 템플릿

**실행 순서**:
Cell 1(환경설정) → Cell 2(동작확인) → Cell 3(MSTAR 압축해제) → Cell 4(gengzhe) → Cell 5(SAMPLE) → Cell 6(Exp A) → Cell 7(Exp B) → Cell 8(Exp C) → Cell 9(Exp D) → Cell 10(커밋)

**GPU 설정**: 상단 메뉴 → 런타임 → 런타임 유형 변경 → T4 GPU 선택

> **주의**: Cell 1의 `TOKEN`을 본인의 GitHub Personal Access Token으로 교체하세요.

> **MSTAR**: `MyDrive/SAR_ATR_Project/data/mstar/`에 ZIP 파일을 올려두세요. Cell 3에서 자동으로 압축 해제 후 실제 구조를 출력합니다. 구조 확인 후 Claude Code에 알려주면 경로를 맞게 수정해드립니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/SAR_ATR_Project'
TOKEN = 'YOUR_GITHUB_TOKEN_HERE'  # Personal Access Token (repo scope)
BRANCH = 'claude/kind-gauss-84kr3m'

import os, sys

!git clone -b {BRANCH} https://{TOKEN}@github.com/cityhunter0831/sar-atr.git /content/repo

%cd /content/repo
!pip install -r requirements.txt -q
!pip install grad-cam scipy -q  # Colab에 누락된 패키지 추가 설치
sys.path.insert(0, '/content/repo')

# Drive 폴더 생성
os.makedirs(f'{DRIVE_ROOT}/data',    exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)

# 실데이터 Drive → /content/data 로 복사 (Drive 직접 학습은 매우 느림)
if not os.path.exists('/content/data'):
    !cp -r {DRIVE_ROOT}/data /content/data
    print('데이터 복사 완료')
else:
    print('데이터 이미 존재 — 스킵')

# 심볼릭 링크 (코드에서 data/ 경로로 접근)
if not os.path.exists('/content/repo/data'):
    os.symlink('/content/data', '/content/repo/data')
if not os.path.exists('/content/repo/results'):
    os.symlink(f'{DRIVE_ROOT}/results', '/content/repo/results')

import torch
print('CUDA:', torch.cuda.is_available())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into '/content/repo'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (26/26), done.
Receiving objects: 100% (37/37), 31.45 KiB | 5.24 MiB/s, done.
remote: Total 37 (delta 7), reused 36 (delta 7), pack-reused 0 (from 0)
Resolving deltas: 100% (7/7), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 15.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 37.8 MB/s eta 0:00:

In [ ]:
# ── Cell 2: 파이프라인 동작 확인 (mock 데이터 — 실데이터 없어도 됨) ──
from core.mock_data import MockSARDataset
from core.models import get_model
from core.train import train_model
from core.interfaces import TrainConfig

train_ds = MockSARDataset(n=200, num_classes=3, seed=0)
val_ds   = MockSARDataset(n=60,  num_classes=3, seed=1)

for model_name in ['smpl', 'resnet18']:
    model  = get_model(model_name, num_classes=3)
    config = TrainConfig(model_name=model_name, num_classes=3, epochs=5, seed=0)
    model, result = train_model(model, train_ds, val_ds, config)
    print(f'[{model_name}] mock val acc = {result.accuracy*100:.1f}%')

print('\n✅ 파이프라인 정상 — 실험 셀로 이동하세요')

In [ ]:
# ── Cell 3: MSTAR ZIP 압축 풀기 + 실제 구조 확인 ────────────────────
import zipfile, os

DRIVE_ROOT  = '/content/drive/MyDrive/SAR_ATR_Project'  # Cell 1 없이도 동작하도록 재정의
MSTAR_DIR   = '/content/data/mstar'
DRIVE_MSTAR = f'{DRIVE_ROOT}/data/mstar'
os.makedirs(MSTAR_DIR, exist_ok=True)

zip_names = [
    'MSTAR-PublicTargetChips-T72-BMP2-BTR70-SLICY.zip',
    'MSTAR-PublicMixedTargets-CD1.zip',
    'MSTAR-PublicMixedTargets-CD2.zip',
]

for fname in zip_names:
    src = f'{DRIVE_MSTAR}/{fname}'
    dst = f'{MSTAR_DIR}/{fname}'
    if not os.path.exists(src):
        print(f'Drive에 없음 (스킵): {fname}')
        continue
    if not os.path.exists(dst):
        print(f'Drive → /content 복사 중: {fname}')
        !cp "{src}" "{dst}"
    print(f'압축 해제 중: {fname}')
    with zipfile.ZipFile(dst) as z:
        z.extractall(MSTAR_DIR)
    print('  → 완료')

# 실제 폴더 구조 출력 (depth 4까지)
print('\n── 실제 구조 ──────────────────────────────')
for root, dirs, files in os.walk(MSTAR_DIR):
    dirs.sort()
    level = root.replace(MSTAR_DIR, '').count(os.sep)
    if level > 4:
        dirs.clear()
        continue
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level == 4:
        subindent = '  ' * (level + 1)
        for f in files[:3]:
            print(f'{subindent}{f}')
        if len(files) > 3:
            print(f'{subindent}... ({len(files)} files total)')
print('────────────────────────────────────────────')
print('위 구조를 Claude Code에 알려주면 경로를 맞게 수정해드립니다.')

# ── SAR-ship ZIP 압축 풀기 ──────────────────────────────────────────
SARSHIP_DIR   = '/content/data/sarship'
DRIVE_SARSHIP = f'{DRIVE_ROOT}/data/sarship'
os.makedirs(SARSHIP_DIR, exist_ok=True)

sarship_zip = f'{DRIVE_SARSHIP}/ship_dataset_v0.zip'
if os.path.exists(sarship_zip):
    print(f'SAR-ship 압축 해제 중: ship_dataset_v0.zip')
    with zipfile.ZipFile(sarship_zip) as z:
        z.extractall(SARSHIP_DIR)
    # 이미지 파일 개수 확인
    import glob
    imgs = glob.glob(f'{SARSHIP_DIR}/**/*.png', recursive=True) + glob.glob(f'{SARSHIP_DIR}/**/*.jpg', recursive=True)
    print(f'  → 이미지 {len(imgs)}장 추출 완료')
else:
    print(f'SAR-ship ZIP 없음 (스킵): {sarship_zip}')
    print('  → Exp D OE는 mock 데이터로 실행됩니다.')

In [ ]:
# ── Cell 4: gengzhe2015 데이터 받기 (실험 A용) ────────────────────────
# Drive에 이미 있으면 건너뛰어도 됨
import os
GENG_DST = f'{DRIVE_ROOT}/data/clutter_gengzhe'

if not os.path.exists(GENG_DST):
    !git clone https://github.com/gengzhe2015/SAR-target-recognition /content/gengzhe2015
    !cp -r /content/gengzhe2015 {GENG_DST}
    !cp -r {GENG_DST} /content/data/clutter_gengzhe
    print('복사 완료')
else:
    if not os.path.exists('/content/data/clutter_gengzhe'):
        !cp -r {GENG_DST} /content/data/clutter_gengzhe
    print('이미 존재함 — 스킵')

!ls /content/data/clutter_gengzhe


In [ ]:
# ── Cell 5: SAMPLE 데이터셋 받기 (실험 C용) ─────────────────────────
# Drive에 이미 있으면 건너뛰어도 됨
import os
SAMPLE_DST = f'{DRIVE_ROOT}/data/sample'

if not os.path.exists(f'{SAMPLE_DST}/png_images'):
    !git clone https://github.com/benjaminlewis-afrl/SAMPLE_dataset_public /content/sample_repo
    os.makedirs(SAMPLE_DST, exist_ok=True)
    !cp -r /content/sample_repo/png_images {SAMPLE_DST}/png_images
    print('SAMPLE 데이터 복사 완료')
else:
    print('이미 존재함 — 스킵')

# /content/data 에도 동기화
if not os.path.exists('/content/data/sample'):
    !cp -r {SAMPLE_DST} /content/data/sample

!ls /content/data/sample/png_images | head -10


In [ ]:
# ── Cell 6: Exp A — 클러터 전이 (Table 4) ────────────────────────────
# 실 데이터 없으면 mock으로 자동 대체됨
import sys; sys.argv = ['']

from experiments.exp_a_clutter_transfer import run_all
results_a = run_all(epochs=60)

In [ ]:
# ── Cell 6b: Exp A — SSIM 경계 아티팩트 정량화 (우리 팀 개선 #1) ────
# clutter transfer 전후 SSIM 측정 → results/exp_a/boundary_ssim.json
from experiments.exp_a_clutter_transfer import run_boundary_ssim_analysis
ssim_result = run_boundary_ssim_analysis(n_samples=20)
print(ssim_result)

In [ ]:
# ── 진단 셀: MSTAR 파일 포맷 확인 ─────────────────────────────────
import sys; sys.path.insert(0, "/content/repo")
%cd /content/repo
!python scripts/diag_mstar_format.py


In [ ]:
# ── Cell 7: Exp B — Phase History 보간 증강 (논문 Section 2.1 재현) ──────────
# MSTAR Mixed Targets CD2 필요. 없으면 mock 데이터로 동작 확인.
import sys; sys.argv = ['']

from experiments.exp_b_ph_scattering import run as run_b
results_b = run_b(model_name='smpl', epochs=60, n_interp=5)

In [ ]:
# ── Cell 8: Exp C — 대비 보정 + Optuna (Figure 1) ────────────────────
# SAMPLE dataset (synthetic→measured) 사용. 데이터 없으면 mock으로 자동 대체.
from experiments.exp_c_contrast_optuna import run as run_c
results_c = run_c(model_name='resnet18', n_optuna_trials=20, epochs_full=60, epochs_trial=10)


In [ ]:
# ── Cell 9: Exp D — OOD 탐지 ─────────────────────────────────────────
from experiments.exp_d_ood import run as run_d
results_d = run_d(model_name='smpl', j_list=[1, 2, 3], epochs=60)

In [ ]:
# ── Cell 10: 결과 커밋 (세션 종료 전 반드시 실행) ────────────────────
%cd /content/repo
import os
os.system("git config user.email 'colab@sar-atr'")
os.system("git config user.name 'Colab Runner'")
os.system('git add results/')
os.system("git commit -m 'exp: Colab 실행 결과 업데이트' || echo 'nothing to commit'")
os.system('git push')